### IMU-GPS learned increments (v2) — Option A
Abraham Rodriguez Vazquez

**Defended method:** the network never sees pose. It maps a window of MARG + GPS velocity to a **pose increment** `(Δpos, Δq)`, then those increments are **integrated** into a trajectory (learned dead reckoning). No Q/R, no Kalman filter at runtime, no ground-truth `prev_state` in the inputs.

**Compared against:** `insfilterMARG` on the same stream (`dataset_v2_filterstate.csv` if you ran `makeDataset_v2.m`), using **ATE** and **RTE**.

**Option B** (recursive net, own pose fed back, scheduled sampling) is *not* trained here — Trabajo Futuro only.

Also kept: chronological split with a gap, train-only normalization, Adam, unit Δq + smooth quat loss, fixed seeds, CUDA-or-CPU.

> Pose columns in the CSV are used **only** to build increment *labels* and the GT path, never as model inputs.

In [ ]:
# --- imports & configuration ---
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 1
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

N_LAGS      = 10
W           = N_LAGS + 1
BATCH_SIZE  = 1000
EPOCHS      = 100
LR          = 1e-3
LAMBDA_QUAT = 1.0
VAL_FRAC    = 0.15
TEST_FRAC   = 0.15
GAP         = N_LAGS
IMU_FS      = 160
WARMUP      = 160
RTE_SEC     = 1.0

DATA_CANDIDATES = ["dataset_v2.csv", "dataset.csv"]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), "dataset.csv")
DIAG_PATH = "dataset_v2_filterstate.csv"
print("device:", device, "| data file:", DATA_PATH, "| EKF diag:", os.path.exists(DIAG_PATH))

In [ ]:
# --- load CSV; pose columns are LABELS only ---
names = ["imu_accx","imu_accy","imu_accz","imu_wx","imu_wy","imu_wz",
         "imu_magx","imu_magy","imu_magz",
         "gps_velx","gps_vely","gps_velz",
         "prev_state_posx","prev_state_posy","prev_state_posz",
         "prev_state_quatw","prev_state_quatx","prev_state_quaty","prev_state_quatz",
         "gt_state_posx","gt_state_posy","gt_state_posz",
         "gt_state_quatw","gt_state_quatx","gt_state_quaty","gt_state_quatz"]

MEAS = names[:12]   # 12 measurement channels — the only network inputs
df = pd.read_csv(DATA_PATH, names=names)
print("dataframe shape:", df.shape)
df[MEAS].head()

In [ ]:
# --- quaternion helpers (w, x, y, z) ---
def q_normalize(q, eps=1e-8):
    return q / (np.linalg.norm(q, axis=-1, keepdims=True) + eps)

def q_conj(q):
    o = q.copy(); o[..., 1:] *= -1; return o

def q_mul(a, b):
    aw, ax, ay, az = [a[..., k] for k in range(4)]
    bw, bx, by, bz = [b[..., k] for k in range(4)]
    return np.stack([
        aw*bw - ax*bx - ay*by - az*bz,
        aw*bx + ax*bw + ay*bz - az*by,
        aw*by - ax*bz + ay*bw + az*bx,
        aw*bz + ax*by - ay*bx + az*bw,
    ], axis=-1)

def q_delta(q_prev, q_curr):
    return q_normalize(q_mul(q_conj(q_normalize(q_prev)), q_normalize(q_curr)))

def q_ang_deg(q_a, q_b):
    dot = np.clip(np.abs((q_normalize(q_a) * q_normalize(q_b)).sum(axis=-1)), -1.0, 1.0)
    return np.degrees(2.0 * np.arccos(dot))

In [ ]:
# --- windows of MEASUREMENTS only; targets are per-step increments ---
meas = df[MEAS].to_numpy(dtype=np.float32)                 # (T, 12)
pos_prev = df[["prev_state_posx","prev_state_posy","prev_state_posz"]].to_numpy(np.float32)
pos_curr = df[["gt_state_posx","gt_state_posy","gt_state_posz"]].to_numpy(np.float32)
q_prev   = df[["prev_state_quatw","prev_state_quatx","prev_state_quaty","prev_state_quatz"]].to_numpy(np.float32)
q_curr   = df[["gt_state_quatw","gt_state_quatx","gt_state_quaty","gt_state_quatz"]].to_numpy(np.float32)

dpos = pos_curr - pos_prev                                  # (T, 3) metres / sample
dquat = q_delta(q_prev, q_curr)                             # (T, 4) relative rotation

T = len(df)
X = np.stack([meas[t-N_LAGS:t+1].T for t in range(N_LAGS, T)], axis=0)  # (N, 12, W)
Y = np.concatenate([dpos[N_LAGS:], dquat[N_LAGS:]], axis=1)             # (N, 7)
pos_gt = pos_curr[N_LAGS:]                                  # absolute GT for integration eval
q_gt   = q_normalize(q_curr[N_LAGS:])
print("X", X.shape, "| Y", Y.shape, "(Δpos + Δq; no pose in X)")

In [ ]:
# --- chronological split with a gap ---
N = len(X)
n_test  = int(N * TEST_FRAC)
n_val   = int(N * VAL_FRAC)
n_train = N - n_val - n_test - 2 * GAP
assert n_train > 0, "not enough samples"

tr = slice(0, n_train)
va = slice(n_train + GAP, n_train + GAP + n_val)
te = slice(n_train + GAP + n_val + GAP, N)

X_tr, Y_tr = X[tr], Y[tr]
X_va, Y_va = X[va], Y[va]
X_te, Y_te = X[te], Y[te]
pos_te, q_te = pos_gt[te], q_gt[te]
print(f"train {len(X_tr)} | val {len(X_va)} | test {len(X_te)}")

In [ ]:
# --- normalize measurements + Δpos on TRAIN only; Δq stays unit ---
feat_mean = X_tr.mean(axis=(0, 2))
feat_std  = X_tr.std(axis=(0, 2)) + 1e-8
dpos_mean = Y_tr[:, :3].mean(axis=0)
dpos_std  = Y_tr[:, :3].std(axis=0) + 1e-8

def norm_X(a):
    a = a.copy()
    a = (a - feat_mean[None, :, None]) / feat_std[None, :, None]
    return a

def norm_Y(a):
    a = a.copy()
    a[:, :3] = (a[:, :3] - dpos_mean) / dpos_std
    a[:, 3:] = q_normalize(a[:, 3:])
    return a

X_tr_n, X_va_n, X_te_n = norm_X(X_tr), norm_X(X_va), norm_X(X_te)
Y_tr_n, Y_va_n, Y_te_n = norm_Y(Y_tr), norm_Y(Y_va), norm_Y(Y_te)
dpos_mean_t = torch.tensor(dpos_mean, dtype=torch.float32)
dpos_std_t  = torch.tensor(dpos_std,  dtype=torch.float32)
print("Δpos scale (m/sample):", dpos_std)

In [ ]:
class IncDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

train_loader = DataLoader(IncDataset(X_tr_n, Y_tr_n), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(IncDataset(X_va_n, Y_va_n), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(IncDataset(X_te_n, Y_te_n), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def unit_quat(q, eps=1e-8):
    return q / (q.norm(dim=1, keepdim=True) + eps)

def quat_loss(q_pred, q_true):
    dot = (unit_quat(q_pred) * unit_quat(q_true)).sum(dim=1)
    return (1.0 - dot ** 2).mean()

def quat_ang_deg_t(q_pred, q_true):
    dot = (unit_quat(q_pred) * unit_quat(q_true)).sum(dim=1).abs().clamp(-1.0, 1.0)
    return torch.rad2deg(2.0 * torch.acos(dot))

In [ ]:
# Conv1d(12) -> LSTM -> MLP -> (Δpos, Δq)
class IncModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1d  = nn.Conv1d(12, 12, kernel_size=3)
        self.lstm    = nn.LSTM(12, 256, num_layers=1, batch_first=True)
        self.flat    = nn.Flatten()
        self.linear1 = nn.Linear(2304, 512); self.relu1 = nn.ReLU()
        self.linear2 = nn.Linear(512, 128);  self.relu2 = nn.ReLU()
        self.linear3 = nn.Linear(128, 7)

    def forward(self, x):
        x = self.conv1d(x)                      # (N,12,W-2)
        out, _ = self.lstm(x.permute(0, 2, 1))  # (N,W-2,256)
        z = self.relu2(self.linear2(self.relu1(self.linear1(self.flat(out)))))
        z = self.linear3(z)
        return torch.cat([z[:, :3], unit_quat(z[:, 3:7])], dim=1)

_o = IncModel()(torch.zeros(4, 12, W))
print("model output:", tuple(_o.shape), "(expected (4, 7))")

In [ ]:
@torch.no_grad()
def evaluate_increments(model, loader):
    model.eval()
    se, n, angs = 0.0, 0, []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        dp = pred[:, :3].cpu() * dpos_std_t + dpos_mean_t
        dt = yb[:, :3].cpu()   * dpos_std_t + dpos_mean_t
        se += ((dp - dt) ** 2).sum().item(); n += dp.shape[0]
        angs.append(quat_ang_deg_t(pred[:, 3:7], yb[:, 3:7]).cpu())
    model.train()
    rmse = (se / (n * 3)) ** 0.5
    ang = torch.cat(angs)
    return rmse, ang.mean().item(), ang.median().item()

In [ ]:
model = IncModel().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=8, factor=0.5)
train_losses, best_val, best_state = [], float("inf"), None

for e in range(EPOCHS):
    model.train(); run = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = nn.functional.mse_loss(pred[:, :3], yb[:, :3]) + LAMBDA_QUAT * quat_loss(pred[:, 3:7], yb[:, 3:7])
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item(); train_losses.append(loss.item())
    rmse, ang_mean, _ = evaluate_increments(model, val_loader)
    val_metric = rmse + ang_mean / 90.0
    sched.step(val_metric)
    if val_metric < best_val:
        best_val = val_metric
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if e % 5 == 0 or e == EPOCHS - 1:
        print(f"epoch {e:3d} | train {run:8.4f} | val Δpos-RMSE {rmse:.4e} m/step | val Δq {ang_mean:.3f} deg")

if best_state is not None:
    model.load_state_dict(best_state)
print("restored best-on-val weights.")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(train_losses, lw=0.6)
plt.xlabel("iteration"); plt.ylabel("train loss"); plt.title("Training loss (increments)")
plt.grid(True, alpha=0.3); plt.show()

rmse, ang_mean, ang_med = evaluate_increments(model, test_loader)
print(f"[TEST increments] Δpos-RMSE {rmse:.4e} m/step | Δq mean {ang_mean:.3f} deg | median {ang_med:.3f} deg")

In [ ]:
# --- integrate predicted increments on the TEST split (one GT seed at t0) ---
@torch.no_grad()
def predict_increments(model, Xn):
    model.eval()
    outs = []
    for i in range(0, len(Xn), BATCH_SIZE):
        xb = torch.tensor(Xn[i:i+BATCH_SIZE], dtype=torch.float32, device=device)
        pred = model(xb).cpu().numpy()
        dp = pred[:, :3] * dpos_std + dpos_mean
        dq = q_normalize(pred[:, 3:])
        outs.append(np.concatenate([dp, dq], axis=1))
    model.train()
    return np.concatenate(outs, axis=0)

inc = predict_increments(model, X_te_n)

pos_hat = np.zeros_like(pos_te)
q_hat   = np.zeros_like(q_te)
pos_hat[0] = pos_te[0]                 # single known start (standard odometry)
q_hat[0]   = q_te[0]
for t in range(1, len(pos_te)):
    pos_hat[t] = pos_hat[t-1] + inc[t, :3]
    q_hat[t]   = q_normalize(q_mul(q_hat[t-1], inc[t, 3:]))

def ate_rte(pos_est, pos_ref, fs=IMU_FS, rte_sec=RTE_SEC):
    err = pos_est - pos_ref
    ate = float(np.sqrt((err ** 2).sum(axis=1).mean()))
    w = int(rte_sec * fs)
    rel = []
    for i in range(0, len(pos_est) - w, max(w // 2, 1)):
        rel.append(np.linalg.norm((pos_est[i+w] - pos_est[i]) - (pos_ref[i+w] - pos_ref[i])))
    rte = float(np.mean(rel)) if rel else float("nan")
    return ate, rte

ate_a, rte_a = ate_rte(pos_hat, pos_te)
ang_path = q_ang_deg(q_hat, q_te)
print(f"[Option A integrated TEST] ATE {ate_a:.3f} m | RTE({RTE_SEC:.0f}s) {rte_a:.3f} m | quat mean {ang_path.mean():.2f} deg")

In [ ]:
# --- same TEST slice vs insfilterMARG (needs dataset_v2_filterstate.csv) ---
ekf_ate = ekf_rte = None
pos_ekf = None
if os.path.exists(DIAG_PATH):
    diag = np.loadtxt(DIAG_PATH, delimiter=",")
    # diag rows are IMU samples 0..loopBound-1; CSV training rows start at WARMUP
    fused_pos = diag[WARMUP:, 1:4].astype(np.float32)
    fused_q   = q_normalize(diag[WARMUP:, 4:8].astype(np.float32))
    if len(fused_pos) >= N_LAGS + n_train + 2 * GAP + n_val + len(pos_te):
        fused_pos = fused_pos[N_LAGS:]
        fused_q   = fused_q[N_LAGS:]
        pos_ekf = fused_pos[te]
        q_ekf   = fused_q[te]
        ekf_ate, ekf_rte = ate_rte(pos_ekf, pos_te)
        ekf_ang = q_ang_deg(q_ekf, q_te)
        print(f"[insfilterMARG TEST]     ATE {ekf_ate:.3f} m | RTE({RTE_SEC:.0f}s) {ekf_rte:.3f} m | quat mean {ekf_ang.mean():.2f} deg")
        print("\nMain comparison table (defend these numbers):")
        print(f"  {'method':<22} {'ATE [m]':>10} {'RTE [m]':>10}")
        print(f"  {'Option A (learned Δ)':<22} {ate_a:10.3f} {rte_a:10.3f}")
        print(f"  {'insfilterMARG (EKF)':<22} {ekf_ate:10.3f} {ekf_rte:10.3f}")
    else:
        print("EKF diag length does not match this CSV; skip comparison.")
else:
    print("No dataset_v2_filterstate.csv — run makeDataset_v2.m to get the EKF baseline on this TEST slice.")

In [ ]:
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(1, 3, 1, projection="3d")
ax.plot(pos_te[:, 0], pos_te[:, 1], pos_te[:, 2], label="GT", lw=1)
ax.plot(pos_hat[:, 0], pos_hat[:, 1], pos_hat[:, 2], label="A integrated", lw=1)
if pos_ekf is not None:
    ax.plot(pos_ekf[:, 0], pos_ekf[:, 1], pos_ekf[:, 2], label="EKF", lw=1)
ax.set_title("TEST trajectory"); ax.legend()

ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(np.linalg.norm(pos_hat - pos_te, axis=1), label="A |err|")
if pos_ekf is not None:
    ax2.plot(np.linalg.norm(pos_ekf - pos_te, axis=1), label="EKF |err|")
ax2.set_title("Position error [m]"); ax2.set_xlabel("test step"); ax2.legend()

ax3 = fig.add_subplot(1, 3, 3)
ax3.plot(ang_path, lw=1)
ax3.set_title("A quaternion error [deg]"); ax3.set_xlabel("test step")
ax3.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "feat_mean": feat_mean, "feat_std": feat_std,
    "dpos_mean": dpos_mean, "dpos_std": dpos_std,
    "n_lags": N_LAGS, "option": "A",
}, "sensorFusion_v2_optionA.pth")
print("saved sensorFusion_v2_optionA.pth")